In [ ]:
import pandas as pd

def get_driver_performance_recent(sessions_list):
    
    if not sessions_list:
        return pd.DataFrame()
    all_race_data = []
    
    for session in sessions_list:
        event_name = session.event['EventName']
        results = session.results.copy()
        
        driver_data = results[["Abbreviation", "Position", "Points"]].copy()
        driver_data.rename(columns={"Abbreviation": "Driver"}, inplace=True)
        driver_data["Event"] = event_name
        
        all_race_data.append(driver_data)
        
    combined_df = pd.concat(all_race_data, ignore_index=True)
    
    performance_summary = combined_df.pivot(
        index="Driver", 
        columns="Event", 
        values=["Position", "Points"]
    )
    
    performance_summary.columns = [f"{col[0]}_{col[1]}" for col in performance_summary.columns]
    
    return performance_summary.reset_index()


def get_constructor_performance_recent(sessions_list):

    if not sessions_list:
        return pd.DataFrame()
        
    all_constructor_data = []
    
    for session in sessions_list:
        event_name = session.event['EventName']
        results = session.results.copy()
        
        team_points = results.groupby("TeamName")["Points"].sum().reset_index()
        team_points.rename(columns={"TeamName": "Constructor"}, inplace=True)
        team_points["Event"] = event_name
        
        all_constructor_data.append(team_points)
        
    combined_df = pd.concat(all_constructor_data, ignore_index=True)
    
    constructor_summary = combined_df.pivot(
        index="Constructor", 
        columns="Event", 
        values="Points"
    )
    
    if isinstance(constructor_summary.columns, pd.MultiIndex):
        constructor_summary.columns = [f"Points_{col[1]}" for col in constructor_summary.columns]
    else:
        constructor_summary.columns = [f"Points_{col}" for col in constructor_summary.columns]
    
    return constructor_summary.reset_index()

In [ ]:
import fastf1

season = 2024
race = 4 

sessions_to_analyze = []

start_round = max(1, race - 2)

for r in range(start_round, race + 1):
    Result = fastf1.get_session(season, r, 'R')
    Result.load()
    sessions_to_analyze.append(Result)

# Generate the performance dataframes
driver_form = get_driver_performance_recent(sessions_to_analyze)
constructor_form = get_constructor_performance_recent(sessions_to_analyze)

driver_form.iloc[:, 4:7] = driver_form.iloc[:, 4:7].fillna(0)
driver_form.iloc[:,1:4] = driver_form.iloc[:,1:4].fillna(22)
driver_form['AveragePositionFromLast3Races'] = driver_form.iloc[:, 1:4].mean(axis=1)
driver_form['AveragePointsFromLast3Races'] = driver_form.iloc[:, 4:7].mean(axis=1)


constructor_form["ConstructorAveragePointFromLast3Races"] = constructor_form.iloc[:,1:4].mean(axis=1)

constructor_form


core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data


req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '16', '81', '14', '63', '38', '4', '44', '27', '23', '20', '31', '2', '22', '3', '77', '24', '18', '10']
core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data


,Constructor,Points_Australian Grand Prix,Points_Japanese Grand Prix,Points_Saudi Arabian Grand Prix,ConstructorAveragePointFromLast3Races
0,Alpine,0.0,0.0,0.0,0.000000
1,Aston Martin,12.0,8.0,10.0,10.000000
2,Ferrari,44.0,27.0,22.0,31.000000
3,Haas F1 Team,3.0,0.0,1.0,1.333333
4,Kick Sauber,0.0,0.0,0.0,0.000000
5,McLaren,27.0,14.0,16.0,19.000000
6,Mercedes,0.0,8.0,10.0,6.000000
7,RB,6.0,1.0,0.0,2.333333
8,Red Bull Racing,10.0,44.0,43.0,32.333333
9,Williams,0.0,0.0,0.0,0.000000


In [ ]:
driver_form.iloc[:, 4:7] = driver_form.iloc[:, 4:7].fillna(0)

In [5]:
driver_form.iloc[:,1:4] = driver_form.iloc[:,1:4].fillna(22)

In [6]:
driver_form['AveragePositionFromLast3Races'] = driver_form.iloc[:, 1:4].mean(axis=1)

In [7]:
driver_form['AveragePointsFromLast3Races'] = driver_form.iloc[:, 4:7].mean(axis=1)

In [8]:
driver_form

,Driver,Position_Australian Grand Prix,Position_Japanese Grand Prix,Position_Saudi Arabian Grand Prix,Points_Australian Grand Prix,Points_Japanese Grand Prix,Points_Saudi Arabian Grand Prix,AveragePositionFromLast3Races,AveragePointsFromLast3Races
0,ALB,11.0,20.0,11.0,0.0,0.0,0.0,14.000000,0.000000
1,ALO,8.0,6.0,5.0,4.0,8.0,10.0,6.333333,7.333333
2,BEA,22.0,22.0,7.0,0.0,0.0,6.0,17.000000,2.000000
3,BOT,14.0,14.0,17.0,0.0,0.0,0.0,15.000000,0.000000
4,GAS,13.0,16.0,20.0,0.0,0.0,0.0,16.333333,0.000000
5,HAM,18.0,9.0,9.0,0.0,2.0,2.0,12.000000,1.333333
6,HUL,9.0,11.0,10.0,2.0,0.0,1.0,10.000000,1.000000
7,LEC,2.0,4.0,3.0,19.0,12.0,16.0,3.000000,15.666667
8,MAG,10.0,13.0,12.0,1.0,0.0,0.0,11.666667,0.333333
9,NOR,3.0,5.0,8.0,15.0,10.0,4.0,5.333333,9.666667


In [9]:
constructor_form["ConstructorAveragePointFromLast3Races"] = constructor_form.iloc[:,1:4].mean(axis=1)

In [10]:
constructor_form

,Constructor,Points_Australian Grand Prix,Points_Japanese Grand Prix,Points_Saudi Arabian Grand Prix,ConstructorAveragePointFromLast3Races
0,Alpine,0.0,0.0,0.0,0.000000
1,Aston Martin,12.0,8.0,10.0,10.000000
2,Ferrari,44.0,27.0,22.0,31.000000
3,Haas F1 Team,3.0,0.0,1.0,1.333333
4,Kick Sauber,0.0,0.0,0.0,0.000000
5,McLaren,27.0,14.0,16.0,19.000000
6,Mercedes,0.0,8.0,10.0,6.000000
7,RB,6.0,1.0,0.0,2.333333
8,Red Bull Racing,10.0,44.0,43.0,32.333333
9,Williams,0.0,0.0,0.0,0.000000
